# Recipe recommender: tái lập kết quả local trên Kaggle

Notebook này dùng `recipe_model/metrics.json` làm nguồn sự thật duy nhất để in bảng so sánh baseline với mô hình chính, biểu đồ Hit@K, heatmap chọn trọng số và các ma trận đánh giá. Vì repository hiện chỉ có model artifacts/metrics, notebook chưa tự huấn luyện lại từ đầu; muốn huấn luyện lại cần bổ sung dataset và script train vào Kaggle Dataset.

**Cách dùng trên Kaggle:** upload thư mục `recipe_model` thành Kaggle Dataset, rồi bật Internet nếu cần cài thêm thư viện. Notebook tự tìm `metrics.json` trong `/kaggle/input`, `/kaggle/working` hoặc thư mục hiện tại.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 120

SEARCH_ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()]
metric_candidates = []
for root in SEARCH_ROOTS:
    if root.exists():
        metric_candidates.extend(root.rglob('metrics.json'))
if not metric_candidates:
    raise FileNotFoundError('Không tìm thấy metrics.json. Hãy upload thư mục recipe_model lên Kaggle Dataset.')
METRICS_PATH = next((p for p in metric_candidates if p.name == 'metrics.json'), metric_candidates[0])
ARTIFACT_DIR = METRICS_PATH.parent
with METRICS_PATH.open(encoding='utf-8') as f:
    metrics = json.load(f)
print(f'Đang dùng: {METRICS_PATH}')
print(f'Split sizes: {metrics.get("split_sizes", {})}')

## 1. Kiểm tra tính nhất quán với local
Các giá trị dưới đây là các mốc kiểm chứng lấy trực tiếp từ kết quả local. Assertion giúp notebook dừng ngay nếu artifact bị thay nhầm.

In [ ]:
expected = {
    'validation_hit10': 0.871,
    'test_main_hit1': 0.7373,
    'test_main_hit5': 0.8289,
    'test_main_hit10': 0.8624,
}
actual = {
    'validation_hit10': metrics['validation_hit10'],
    'test_main_hit1': metrics['test']['Deep hybrid (validation selected)']['Hit@1'],
    'test_main_hit5': metrics['test']['Deep hybrid (validation selected)']['Hit@5'],
    'test_main_hit10': metrics['test']['Deep hybrid (validation selected)']['Hit@10'],
}
for key, value in expected.items():
    assert np.isclose(actual[key], value, atol=1e-4), f'{key}: {actual[key]} != {value}'
print('PASS: các mốc local khớp trong sai số 1e-4')
display(pd.DataFrame({'Expected local': expected, 'Loaded artifact': actual}))

## 2. Bảng so sánh baseline và mô hình chính

In [ ]:
test = metrics['test']
display_order = [
    'Random', 'CBF', 'Encoder', 'Jaccard Similarity', 'TF-IDF Cosine',
    'Previous hybrid 50/50', 'Deep hybrid with main ingredient penalty',
    'Deep hybrid (validation selected)'
]
comparison = pd.DataFrame(test).T.reset_index(names='Method')
comparison = comparison[comparison['Method'].isin(display_order)].copy()
comparison['order'] = comparison['Method'].map({name: i for i, name in enumerate(display_order)})
comparison = comparison.sort_values('order').drop(columns='order')
main_name = 'Deep hybrid (validation selected)'
baseline_name = 'Previous hybrid 50/50'
main = comparison.loc[comparison.Method.eq(main_name)].iloc[0]
baseline = comparison.loc[comparison.Method.eq(baseline_name)].iloc[0]
for k in ['Hit@1', 'Hit@5', 'Hit@10']:
    comparison[f'Gain vs {baseline_name} ({k})'] = np.nan
    comparison.loc[comparison.Method.eq(main_name), f'Gain vs {baseline_name} ({k})'] = main[k] - baseline[k]
styled = comparison.style.format({c: '{:.4f}' for c in comparison.columns if c != 'Method'}).background_gradient(subset=['Hit@1','Hit@5','Hit@10'], cmap='Blues')
display(styled)
print('Mô hình chính:', main_name)
print('Tăng so với baseline 50/50:', {k: round(float(main[k]-baseline[k]), 4) for k in ['Hit@1','Hit@5','Hit@10']})

## 3. Biểu đồ Hit@K

In [ ]:
plot_df = comparison.set_index('Method')[['Hit@1','Hit@5','Hit@10']]
ax = plot_df.plot(kind='bar', figsize=(14, 5), color=['#4C78A8','#F58518','#54A24B'])
ax.set_title('So sánh hiệu năng recommendation trên test set')
ax.set_ylabel('Hit rate')
ax.set_xlabel('Phương pháp')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=35)
ax.legend(title='Metric')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, metric in zip(axes, ['Hit@1','Hit@5','Hit@10']):
    vals = plot_df[metric].sort_values()
    colors = ['#D62728' if n == main_name else '#4C78A8' for n in vals.index]
    vals.plot.barh(ax=ax, color=colors)
    ax.set_title(metric); ax.set_xlim(0, 1); ax.set_xlabel('Hit rate')
plt.tight_layout(); plt.show()

## 4. Ma trận kết quả và heatmap chọn trọng số
Với bài toán ranking, confusion matrix cổ điển không phản ánh đúng Hit@K. Vì vậy notebook dùng ma trận phương pháp × Hit@K và ma trận validation weight × Hit@K. Nếu CSV manual review có nhãn, cell cuối sẽ dựng thêm confusion matrix `relevant`/`can_cook`. 

In [ ]:
matrix = plot_df.copy()
plt.figure(figsize=(10, 6))
sns.heatmap(matrix, annot=True, fmt='.4f', cmap='YlGnBu', vmin=0, vmax=1)
plt.title('Ma trận hiệu năng: phương pháp × Hit@K'); plt.xlabel('Metric'); plt.ylabel('Method')
plt.tight_layout(); plt.show()

val = metrics['gnn']['recommendation']['validation']
weight_matrix = pd.DataFrame(val).T[['Hit@1','Hit@5','Hit@10']]
weight_matrix.index.name = 'Graph blend weight'
plt.figure(figsize=(7, 3.5))
sns.heatmap(weight_matrix, annot=True, fmt='.4f', cmap='Oranges', vmin=0, vmax=1)
plt.title('Validation: chọn trọng số graph blend'); plt.tight_layout(); plt.show()
print('Selected weight:', metrics['gnn']['recommendation']['selected_weight'])

In [ ]:
review_path = ARTIFACT_DIR / 'manual_review.csv'
if review_path.exists():
    review = pd.read_csv(review_path)
    labels = review[['relevant_0_or_1', 'can_cook_0_or_1']].apply(pd.to_numeric, errors='coerce').dropna()
    if len(labels):
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(labels['relevant_0_or_1'], labels['can_cook_0_or_1'], labels=[0,1])
        display(pd.DataFrame(cm, index=['Relevant=0','Relevant=1'], columns=['Can cook=0','Can cook=1']))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False)
        plt.title('Confusion matrix: relevant vs can_cook'); plt.ylabel('Relevant'); plt.xlabel('Can cook'); plt.show()
    else:
        print('manual_review.csv chưa có nhãn số ở relevant_0_or_1/can_cook_0_or_1; bỏ qua confusion matrix.')
else:
    print('Không tìm thấy manual_review.csv; bỏ qua confusion matrix.')

## 5. Tóm tắt để đối chiếu báo cáo
- Validation Hit@10: **0.8710**.
- Test mô hình chính Deep hybrid (validation selected): **Hit@1 = 0.7373, Hit@5 = 0.8289, Hit@10 = 0.8624**.
- Trọng số được chọn: **0.0**; vì vậy `test_baseline` và `test_graph_blend` trong artifact local giống nhau.
- AUC/AP link prediction được đọc từ `metrics.json`, không tự suy diễn từ ranking metrics.